In [1]:
!pip install hnswlib

In [2]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

# ===========================
# 1. Buat data 2D acak
# ===========================
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

# ===========================
# 2. Exact NN (Brute Force)
# ===========================
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

# ===========================
# 3. HNSW
# ===========================
# Inisialisasi index HNSW
p = hnswlib.Index(space='l2', dim=dim)  

# Ukuran maksimum elemen yang bisa ditampung
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Tambahkan data
p.add_items(data)

# Set parameter pencarian
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

=== Exact NN ===
Indices: [[959 120 209 573 907]]
Distances: [[0.01866782 0.02255644 0.02365222 0.0251334  0.03423588]]
Waktu: 2.19520902633667 detik

=== HNSW ===
Indices: [[959 120 209 573 907]]
Distances: [[0.00034849 0.00050879 0.00055943 0.00063169 0.0011721 ]]
Waktu: 0.0001785755157470703 detik


| Jumlah Data | Dimensi | Metrik | Hasil Index (Exact vs HNSW) | Waktu Exact (s) | Waktu HNSW (s) |
| :---: | :---: | :---: | :--- | :---: | :---: |
| 1.000 | 2D | L2 | [137, 170, 750, 316, 991], [137, 170, 750, 316, 991] | 0.004155 | 0.000051 |
| 1.000 | 5D | L2 | [895, 451, 375, 386, 411], [895, 451, 375, 386, 411] | 0.003682 | 0.000055 |
| 1.000.000 | 2D | L2 | [286942, 583487, 695123, 948064, 829288], [286942, 583487, 695123, 948064, 829288] | 0.011672 | 0.000136 |
| 1.000.000 | 5D | L2 | [794791, 501340, 912100, 84056, 660781], [794791, 501340, 912100, 84056, 660781] | 0.014260 | 0.000151 |